<a href="https://colab.research.google.com/github/MoeTahech/real-fake-face-detection-thesis/blob/main/notebooks/06_mixed_source_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Phase 6: Mixed-Source Training

Tests whether the Phase 3 generalization gap (near-chance performance on
unseen face-swap deepfakes) is fixable with **training data diversity**,
since Phase 4 showed it is not fixable with architecture/ensembling alone.

Trains a fresh ResNet18 (same fine-tuning recipe as Phase 2: unfreeze `layer4`)
on a **combined** dataset built from both sources used so far:
- StyleGAN-generated faces (140k Real and Fake Faces)
- Face-swap deepfake video frames (Deepfake and Real Images)

Then evaluates the resulting model on three test sets, compared directly
against the Phase 2 (single-source) model:
1. StyleGAN-only test set
2. Face-swap-only test set
3. Combined test set

Runtime → Change runtime type → GPU, then run cells top to bottom.

### Step 1 — Confirm you have a GPU

In [ ]:
!nvidia-smi

Sun Aug  9 20:48:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/thesis_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoints will be saved to / loaded from:", CHECKPOINT_DIR)

Mounted at /content/drive
Checkpoints will be saved to / loaded from: /content/drive/MyDrive/thesis_checkpoints


### Step 3 — Kaggle API access (via Colab Secrets)

Requires secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` already set up (🔑 icon,
left sidebar) with your rotated Kaggle credentials.

In [ ]:
import os

os.environ["KAGGLE_API_TOKEN"] = "YOUR_KAGGLE_API_TOKEN_HERE"

### Step 4 — Download both datasets

In [ ]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces
!unzip -q 140k-real-and-fake-faces.zip -d data_raw

!kaggle datasets download -d manjilkarki/deepfake-and-real-images
!unzip -q deepfake-and-real-images.zip -d data_raw_2
print("Both datasets downloaded.")

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
100% 3.75G/3.75G [00:28<00:00, 141MB/s]

Dataset URL: https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images
License(s): unknown
100% 1.68G/1.68G [00:17<00:00, 101MB/s] 

Both datasets downloaded.


### Step 5 — Auto-locate train/test splits in both datasets

Same robust folder-detection approach used in earlier notebooks — searches
for folders containing `real`/`fake` subfolders (case-insensitive), preferring
ones with "train" or "test" in their path.

In [ ]:
import os

def find_real_fake_folders(root):
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        lower = [d.lower() for d in dirnames]
        if "real" in lower and "fake" in lower:
            candidates.append(dirpath)
    return candidates

def find_split(root, keyword):
    candidates = find_real_fake_folders(root)
    matches = [x for x in candidates if keyword in x.lower()]
    return matches[0] if matches else (candidates[0] if candidates else None)

# Source A: StyleGAN dataset
sourceA_train = find_split("data_raw", "train")
sourceA_test = find_split("data_raw", "test")
print("Source A (StyleGAN) train:", sourceA_train)
print("Source A (StyleGAN) test: ", sourceA_test)

# Source B: face-swap dataset
sourceB_train = find_split("data_raw_2", "train")
sourceB_test = find_split("data_raw_2", "test")
print("Source B (face-swap) train:", sourceB_train)
print("Source B (face-swap) test: ", sourceB_test)

Source A (StyleGAN) train: data_raw/real_vs_fake/real-vs-fake/train
Source A (StyleGAN) test:  data_raw/real_vs_fake/real-vs-fake/test
Source B (face-swap) train: data_raw_2/Dataset/Train
Source B (face-swap) test:  data_raw_2/Dataset/Test


### Step 6 — Build the mixed training set

Combines a capped, class-balanced number of images from **both** sources into
a single `data_mixed/train` and `data_mixed/val` (symlinks, not copies — fast,
no extra disk usage). Folder names are normalized to lowercase `real`/`fake`
regardless of how each source names them.

Capped at 15,000 train / 2,000 val images per class per source (so up to
30,000 train / 4,000 val images per class total) — large enough to be
meaningful, small enough to train in reasonable time on a T4.

In [ ]:
import random

def find_class_dir(split_root, class_name):
    """Case-insensitively find the subfolder matching 'real' or 'fake'."""
    for d in os.listdir(split_root):
        if d.lower() == class_name:
            return os.path.join(split_root, d)
    raise FileNotFoundError(f"No '{class_name}' folder found in {split_root}")

def build_mixed_split(sources, dst_root, n_train_per_class_per_source, n_val_per_class_per_source, seed=42):
    rng = random.Random(seed)
    for cls in ["real", "fake"]:
        train_dst = os.path.join(dst_root, "train", cls)
        val_dst = os.path.join(dst_root, "val", cls)
        os.makedirs(train_dst, exist_ok=True)
        os.makedirs(val_dst, exist_ok=True)

        for source_root in sources:
            cls_dir = find_class_dir(source_root, cls)
            files = os.listdir(cls_dir)
            rng.shuffle(files)
            n_total = n_train_per_class_per_source + n_val_per_class_per_source
            chosen = files[:n_total]
            train_files = chosen[:n_train_per_class_per_source]
            val_files = chosen[n_train_per_class_per_source:n_total]

            source_tag = os.path.basename(os.path.normpath(source_root))
            for fname in train_files:
                src = os.path.abspath(os.path.join(cls_dir, fname))
                dst = os.path.join(train_dst, f"{source_tag}_{fname}")
                if not os.path.exists(dst):
                    os.symlink(src, dst)
            for fname in val_files:
                src = os.path.abspath(os.path.join(cls_dir, fname))
                dst = os.path.join(val_dst, f"{source_tag}_{fname}")
                if not os.path.exists(dst):
                    os.symlink(src, dst)

build_mixed_split(
    sources=[sourceA_train, sourceB_train],
    dst_root="data_mixed",
    n_train_per_class_per_source=15000,
    n_val_per_class_per_source=2000,
)

print("train/real:", len(os.listdir("data_mixed/train/real")))
print("train/fake:", len(os.listdir("data_mixed/train/fake")))
print("val/real:", len(os.listdir("data_mixed/val/real")))
print("val/fake:", len(os.listdir("data_mixed/val/fake")))

train/real: 30000
train/fake: 30000
val/real: 4000
val/fake: 4000


### Step 7 — Write the mixed-source training script

Identical recipe to Phase 2 (unfreeze `layer4` of ResNet18) — the only
difference from Phase 2 is the training data itself, which isolates data
diversity as the variable being tested.

In [ ]:
%%writefile train_mixed.py
"""
Fine-tune ResNet18 (unfreeze layer4) on a mixed-source dataset
(StyleGAN + face-swap combined).

Usage:
    python train_mixed.py --data_dir ./data_mixed --epochs 5 --batch_size 64 --lr 1e-4 --checkpoint_dir .
"""
import argparse
import os
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm


def get_dataloaders(data_dir, batch_size, img_size=224):
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ])
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        normalize,
    ])
    train_set = datasets.ImageFolder(f"{data_dir}/train", transform=train_transform)
    val_set = datasets.ImageFolder(f"{data_dir}/val", transform=val_transform)
    print("Class mapping:", train_set.class_to_idx)
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, val_loader, train_set.class_to_idx


def build_model():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model


def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in tqdm(loader, desc="train" if train else "val"):
            images, labels = images.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data_dir", type=str, default="./data_mixed")
    parser.add_argument("--epochs", type=int, default=5)
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--checkpoint_dir", type=str, default=".")
    args = parser.parse_args()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    train_loader, val_loader, class_to_idx = get_dataloaders(args.data_dir, args.batch_size)
    model = build_model().to(device)
    criterion = nn.CrossEntropyLoss()

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable_params)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {n_trainable:,} / {n_total:,}")

    optimizer = torch.optim.Adam(trainable_params, lr=args.lr)

    os.makedirs(args.checkpoint_dir, exist_ok=True)
    save_path = os.path.join(args.checkpoint_dir, "best_model_mixed.pth")

    best_val_acc = 0.0
    for epoch in range(args.epochs):
        print(f"\nEpoch {epoch + 1}/{args.epochs}")
        train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
        val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
        print(f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({"model_state": model.state_dict(), "class_to_idx": class_to_idx}, save_path)
            print(f"Saved new best model to {save_path} (val_acc={val_acc:.4f})")

    print(f"\nDone. Best validation accuracy: {best_val_acc:.4f}")
    print(f"Final checkpoint is at: {save_path}")


if __name__ == "__main__":
    main()


Writing train_mixed.py


### Step 8 — Train on the mixed dataset

In [ ]:
!python train_mixed.py --data_dir ./data_mixed --epochs 5 --batch_size 64 --lr 1e-4 --checkpoint_dir "{CHECKPOINT_DIR}"

Using device: cuda
Class mapping: {'fake': 0, 'real': 1}
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 151MB/s]
Trainable parameters: 8,394,754 / 11,177,538

Epoch 1/5
train: 100% 938/938 [03:25<00:00,  4.57it/s]
val: 100% 125/125 [00:25<00:00,  4.89it/s]
train_loss=0.2314 train_acc=0.9012 val_loss=0.1405 val_acc=0.9403
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/best_model_mixed.pth (val_acc=0.9403)

Epoch 2/5
train: 100% 938/938 [03:13<00:00,  4.84it/s]
val: 100% 125/125 [00:23<00:00,  5.23it/s]
train_loss=0.1014 train_acc=0.9610 val_loss=0.1182 val_acc=0.9553
Saved new best model to /content/drive/MyDrive/thesis_checkpoints/best_model_mixed.pth (val_acc=0.9553)

Epoch 3/5
train: 100% 938/938 [03:02<00:00,  5.15it/s]
val: 100% 125/125 [00:25<00:00,  4.94it/s]
train_loss=0.0666 train_acc=0.9752 val_loss=0.1428 val_acc=0.9477

Epoch 4/5
train: 100%

### Step 9 — Build the three evaluation test sets

Capped at 5,000 images/class per source (or fewer if the source has less) to
keep evaluation time reasonable, using each source's actual test split
(images never used in training).

In [ ]:
def build_eval_subset(source_root, dst_root, n_per_class, seed=123):
    rng = random.Random(seed)
    for cls in ["real", "fake"]:
        dst = os.path.join(dst_root, cls)
        os.makedirs(dst, exist_ok=True)
        cls_dir = find_class_dir(source_root, cls)
        files = os.listdir(cls_dir)
        rng.shuffle(files)
        for fname in files[:n_per_class]:
            src = os.path.abspath(os.path.join(cls_dir, fname))
            dst_path = os.path.join(dst, fname)
            if not os.path.exists(dst_path):
                os.symlink(src, dst_path)

build_eval_subset(sourceA_test, "eval_styleGAN", n_per_class=5000)
build_eval_subset(sourceB_test, "eval_faceswap", n_per_class=5000)
# Combined: just point ImageFolder-style evaluation at both dirs separately and merge results manually (done in Step 10)

print("eval_styleGAN/real:", len(os.listdir("eval_styleGAN/real")))
print("eval_styleGAN/fake:", len(os.listdir("eval_styleGAN/fake")))
print("eval_faceswap/real:", len(os.listdir("eval_faceswap/real")))
print("eval_faceswap/fake:", len(os.listdir("eval_faceswap/fake")))

eval_styleGAN/real: 5000
eval_styleGAN/fake: 5000
eval_faceswap/real: 5000
eval_faceswap/fake: 5000


### Step 10 — Evaluate Phase 2 (single-source) vs. Phase 6 (mixed-source) on both test sets

This is the key comparison: does mixed-source training actually close the
generalization gap Phase 3 found, or only partially help, or not at all?

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), normalize])


def load_resnet(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device).eval()
    return model, checkpoint["class_to_idx"]


def evaluate(model, class_to_idx, test_dir):
    real_idx = class_to_idx["real"]
    test_set = datasets.ImageFolder(test_dir, transform=transform)
    loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=2)

    remap = {idx: class_to_idx[name.lower()] for name, idx in test_set.class_to_idx.items()}

    all_labels, all_probs, all_preds = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)[:, real_idx].cpu().numpy()
            preds = logits.argmax(dim=1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(preds)
            all_labels.extend([remap[l.item()] for l in labels])

    acc = accuracy_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    return acc, auc


phase2_model, phase2_map = load_resnet(f"{CHECKPOINT_DIR}/best_model_finetuned.pth")
phase6_model, phase6_map = load_resnet(f"{CHECKPOINT_DIR}/best_model_mixed.pth")

print(f"{'Model':<25} {'Test set':<15} {'Accuracy':<10} {'AUC':<10}")
results = {}
for model_name, model, cmap in [("Phase 2 (single-source)", phase2_model, phase2_map),
                                  ("Phase 6 (mixed-source)", phase6_model, phase6_map)]:
    for test_name, test_dir in [("StyleGAN", "eval_styleGAN"), ("Face-swap", "eval_faceswap")]:
        acc, auc = evaluate(model, cmap, test_dir)
        results[(model_name, test_name)] = (acc, auc)
        print(f"{model_name:<25} {test_name:<15} {acc:<10.4f} {auc:<10.4f}")

print("\n--- Direct comparison on Face-swap (the previously-unseen domain) ---")
p2_acc, p2_auc = results[("Phase 2 (single-source)", "Face-swap")]
p6_acc, p6_auc = results[("Phase 6 (mixed-source)", "Face-swap")]
print(f"Phase 2: Accuracy={p2_acc:.4f}  AUC={p2_auc:.4f}")
print(f"Phase 6: Accuracy={p6_acc:.4f}  AUC={p6_auc:.4f}")
print(f"Change:  Accuracy={p6_acc-p2_acc:+.4f}  AUC={p6_auc-p2_auc:+.4f}")

Model                     Test set        Accuracy   AUC       
Phase 2 (single-source)   StyleGAN        0.9868     0.9992    
Phase 2 (single-source)   Face-swap       0.4978     0.5245    
Phase 6 (mixed-source)    StyleGAN        0.9549     0.9950    
Phase 6 (mixed-source)    Face-swap       0.8242     0.9283    

--- Direct comparison on Face-swap (the previously-unseen domain) ---
Phase 2: Accuracy=0.4978  AUC=0.5245
Phase 6: Accuracy=0.8242  AUC=0.9283
Change:  Accuracy=+0.3264  AUC=+0.4038
